# Matched Pythia continuation

This notebook performs the causal test: baseline and CPS-prescribed forks start from identical weights and consume identical batches in identical order. Only the declared optimizer control differs.

## Acceptance logic

A successful intervention should improve a preregistered stability or loss criterion without changing token budget. One short continuation is a mechanism test; multiple checkpoints and seeds are needed for a general optimizer claim.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import show_environment
runtime = show_environment()

## Stage 1 — resolve the continuation contract

Environment variables allow the Colab CLI to inject the checkpoint, number of steps, and selected control without editing the notebook.

In [ ]:
import os
from cps.notebook import show_config
from cps.pythia.continuation import ContinuationConfig, ContinuationControl

config = ContinuationConfig(
    revision=os.environ.get("CPS_REVISION", "step1000"),
    steps=int(os.environ.get("CPS_CONTINUATION_STEPS", "20")),
    intervention=ContinuationControl(
        name="cps",
        learning_rate_scale=float(os.environ.get("CPS_LR_SCALE", "0.8")),
        beta1=float(os.environ["CPS_BETA1"]) if "CPS_BETA1" in os.environ else None,
        beta2=float(os.environ["CPS_BETA2"]) if "CPS_BETA2" in os.environ else None,
        epsilon_scale=float(os.environ.get("CPS_EPSILON_SCALE", "1.0")),
        gradient_clip=float(os.environ["CPS_GRADIENT_CLIP"]) if "CPS_GRADIENT_CLIP" in os.environ else None,
    ),
    output_dir="/content/cps-artifacts/continuation",
    verbose=True,
)
show_config(config)

## Stage 2 — run both forks

The live log reports loss and gradient norm at every step for each fork. This makes divergence, stalls, or numerical spikes visible in remote `colab-cli log` output.

In [ ]:
from cps.pythia.continuation import run_matched_continuation

result_path = run_matched_continuation(config)
print(f"[CONTINUATION] evidence={result_path}", flush=True)

## Stage 3 — compare trajectories

The plot is paired by construction: step *k* in both forks used the same token batch.

In [ ]:
import json, pathlib, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

payload=json.loads(pathlib.Path(result_path).read_text())
frames=[]
summary_rows=[]
for name,result in payload["results"].items():
    frame=pd.DataFrame(result["records"])
    frame["fork"]=name
    frames.append(frame)
    summary_rows.append({
        "fork": name,
        "final loss": result["final_loss"],
        "maximum gradient norm": result["maximum_gradient_norm"],
        "loss spike": result["loss_spike"],
    })
trajectory=pd.concat(frames, ignore_index=True)
display(pd.DataFrame(summary_rows))
axis=trajectory.pivot(index="step", columns="fork", values="loss").plot(marker="o", figsize=(10,5))
axis.set_title("Matched continuation loss")
axis.set_ylabel("loss")
axis.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()
axis=trajectory.pivot(index="step", columns="fork", values="gradient_norm").plot(figsize=(10,5))
axis.set_title("Matched continuation gradient norm")
axis.set_ylabel("||g||₂")
axis.grid(True, alpha=0.25)
plt.tight_layout(); plt.show()

## Decision discipline

Report both forks, all preregistered metrics, and any failures. Do not redefine the success criterion after seeing the curves. A negative intervention result is useful: it rejects a planner surrogate or reveals that the reduced local operator did not control the realized continuation.

In [ ]:
from cps.notebook import export_artifacts
archive = export_artifacts()
print(f"Artifact archive ready for colab-cli download: {archive}", flush=True)